## Objective

This notebook transforms transaction-level data into a customer-centric analytical dataset that captures historical purchasing behavior in a form suitable for segmentation and CLV modeling.

The process follows a two-stage aggregation:

1. **Transactions → Orders:** Individual line items are collapsed into order-level summaries to prevent double-counting within multi-item invoices.
2. **Orders → Customers:** Order summaries are aggregated into customer-level features capturing RFM metrics and behavioral indicators.

The resulting Customer 360 table contains one row per customer with features summarizing purchasing patterns, value, and engagement.

### Time Anchor

A consistent snapshot date is defined as the day after the final transaction in the dataset. This anchors all time-based calculations (recency, tenure, age) to a single point, preventing temporal leakage and enabling fair comparison across customers.

### Notebook Outputs

Two customer-level datasets are produced:

1. **Customer features table** containing RFM and behavioral features for modeling and segmentation.
2. **RFM-only table** for scoring, interpretability, and business-facing segmentation.

Both outputs are persisted as reusable artifacts for downstream CLV modeling, customer segmentation, and BI reporting.

## Data Import and Reference Date

In [ ]:
import numpy as np
import pandas as pd

# Load the transaction table from the cleaning notebook
file_path = "/kaggle/input/datasets/online-retail-clean/transactions_clean.csv"
txn_data = pd.read_csv(file_path, parse_dates = ["InvoiceDate"])

# Define the analysis cutoff as the day after the final transaction
# This ensures all customers have equal opportunity to be seen as active up to this point
snapshot_date = txn_data["InvoiceDate"].max() + pd.Timedelta(days = 1)
print(f"Snapshot date: {snapshot_date}")
print(f"Transaction rows: {len(txn_data):,}")
print(f"Unique customers in transactions: {txn_data['CustomerID'].nunique():,}")

Snapshot date: 2011-12-10 12:50:00
Transaction rows: 804,008
Unique customers in transactions: 5,840


## Two-Stage Aggregation: Transactions to Orders to Customers

Retail transaction data typically contains multiple line items per order (invoice). Aggregating directly from transactions to customers would double-count customers who made multi-item purchases. The two-stage approach first collapses line items into order-level summaries, then aggregates orders into customer features.

**Stage 1: Transactions → Orders**

Each invoice is summarized by its purchase date, total revenue, and total items. This aligns with how businesses measure behavior: orders, revenue per order, and basket size.

**Stage 2: Orders → Customers**

Order summaries are aggregated into customer-level features:

- **Frequency:** Number of distinct orders.
- **Monetary:** Total revenue across all orders.
- **Average Order Value (AOV):** Mean revenue per order.
- **Average Basket Size:** Mean items per order.
- **First/Last Purchase:** Temporal bounds of the customer relationship.

**Time-Based Features**

All time-based features are anchored to the snapshot date:

- **RecencyDays:** Days since most recent purchase.
- **TenureDays:** Duration between first and last purchase.
- **CustomerAgeDays:** Duration between first purchase and snapshot.
- **PurchaseRateMonthly:** Orders per month, calculated as Frequency divided by tenure in months.

In [2]:
# Stage 1: Collapse line items into order-level summaries
# This prevents double-counting when customers purchase multiple items in one invoice
orders = txn_data.groupby(["CustomerID", "Invoice"], as_index = False).agg(
    OrderDate = ("InvoiceDate", "max"),
    OrderRevenue = ("TotalPrice", "sum"),
    Items = ("Quantity", "sum")
)

print(f"Order-level records: {len(orders):,}")
print(f"Distinct customers at order level: {orders['CustomerID'].nunique():,}")

# Stage 2: Aggregate orders into customer-level features
customers = orders.groupby("CustomerID", as_index = False).agg(
    Frequency = ("Invoice", "nunique"),
    Monetary = ("OrderRevenue", "sum"),
    AvgOrderValue = ("OrderRevenue", "mean"),
    AvgItemsPerOrder = ("Items", "mean"),
    FirstPurchase = ("OrderDate", "min"),
    LastPurchase = ("OrderDate", "max")
)

# Calculate time-based features relative to the snapshot date
customers["RecencyDays"] = (snapshot_date - customers["LastPurchase"]).dt.days

# Tenure: active purchasing window (first to last purchase)
customers["TenureDays"] = (customers["LastPurchase"] - customers["FirstPurchase"]).dt.days

# Customer age: time since first purchase relative to snapshot
customers["CustomerAgeDays"] = (snapshot_date - customers["FirstPurchase"]).dt.days

# Purchase rate: orders per month
# For customers with zero tenure (same-day first and last purchase), 
# set PurchaseRate = Frequency to avoid division by zero
customers["PurchaseRateMonthly"] = customers["Frequency"] / ((customers["TenureDays"] / 30) + (1/30))
customers.loc[customers["TenureDays"] == 0, "PurchaseRateMonthly"] = customers.loc[customers["TenureDays"] == 0, "Frequency"]

# Validate against cleaning notebook customer count
expected_customers = 5840
actual_customers = len(customers)
print(f"\nCustomer validation:")
print(f"Expected customers (from cleaning): {expected_customers:,}")
print(f"Actual customers in feature table: {actual_customers:,}")
print(f"Match: {expected_customers == actual_customers}")

Order-level records: 36,591
Distinct customers at order level: 5,840

Customer validation:
Expected customers (from cleaning): 5,840
Actual customers in feature table: 5,840
Match: True


In [ ]:
# Show feature summary statistics for validation
feature_summary = customers[["Frequency", "Monetary", "AvgOrderValue", "AvgItemsPerOrder", 
                              "RecencyDays", "TenureDays", "CustomerAgeDays", "PurchaseRateMonthly"]].describe()

feature_summary.round(2)

,Frequency,Monetary,AvgOrderValue,AvgItemsPerOrder,RecencyDays,TenureDays,CustomerAgeDays,PurchaseRateMonthly
count,5840.00,5840.00,5840.00,5840.00,5840.00,5840.00,5840.00,5840.00
mean,6.27,2827.61,357.32,210.98,200.49,273.40,474.25,0.98
std,12.77,13308.57,367.73,228.65,208.76,258.52,223.32,1.79
min,1.00,2.90,2.90,1.00,1.00,0.00,1.00,0.08
25%,1.00,343.90,179.35,94.00,26.00,0.00,311.75,0.39
50%,3.00,879.39,282.22,157.76,95.00,221.00,530.00,0.81
75%,7.00,2253.76,416.05,261.00,379.00,511.00,668.00,1.00
max,376.00,533895.83,8951.26,4803.00,739.00,738.00,739.00,60.00


## Feature Validation and Quality Check

The customer feature table contains 5,840 customers, matching exactly the count from the cleaning notebook. This confirms that no customers were lost or gained during the feature engineering process.

Key observations from the feature distributions:

- **Frequency** is heavily right-skewed (median 3 orders, max 376).
- **Monetary** shows extreme range (median £879, max £533,896).
- **RecencyDays** spans 1-739 days, reflecting the full 2-year observation window.
- **TenureDays** has a median of 221 days, indicating most customer relationships span several months.
- **CustomerAgeDays** shows a median of 530 days, confirming the dataset captures long-term customer histories.
- **PurchaseRateMonthly** has a median of 0.81 orders per month, with the expected pattern after correcting for zero-tenure inflation.

These distributions will inform outlier handling and transformation choices in subsequent modeling notebooks. The extreme skewness in Monetary and Frequency suggests log transformations may be appropriate for CLV modeling.

## Export for Downstream Use

The engineered customer features and RFM table are exported as reusable datasets to support downstream modeling, segmentation, and BI workflows. Persisting these customer-level tables ensures that subsequent analyses operate on a consistent and validated feature set, avoiding repeated feature construction and maintaining alignment across notebooks and dashboards.

In [4]:
# Export Customer 360 feature table
customers_file = "customers_features.csv"
customers.to_csv(customers_file, index = False)

# Export RFM-only table for scoring and interpretability
rfm_file = "customers_rfm.csv"
rfm_columns = ["CustomerID", "RecencyDays", "Frequency", "Monetary"]
rfm = customers[rfm_columns].copy()
rfm.to_csv(rfm_file, index=False)

print(f"Customer features saved to: {customers_file}")
print(f"RFM table saved to: {rfm_file}")
print(f"\nFeature table shape: {customers.shape}")
print(f"RFM table shape: {rfm.shape}")
print(f"\nFeature columns:")
for col in customers.columns:
    print(f"  - {col}")

Customer features saved to: customers_features.csv
RFM table saved to: customers_rfm.csv

Feature table shape: (5840, 11)
RFM table shape: (5840, 4)

Feature columns:
  - CustomerID
  - Frequency
  - Monetary
  - AvgOrderValue
  - AvgItemsPerOrder
  - FirstPurchase
  - LastPurchase
  - RecencyDays
  - TenureDays
  - CustomerAgeDays
  - PurchaseRateMonthly
